# Attention Mechanism Basics

Scaled dot-product attention and single-head self-attention — implemented from scratch
with full backpropagation, validated against PyTorch, and tested on a sequence
classification task where a learnable query must find a marked position.

We answer four concrete questions with numbers:
1. What do Query, Key, and Value mean in $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$?
2. Do from-scratch attention gradients match PyTorch's `scaled_dot_product_attention`?
3. Does a causal mask block attention to future positions?
4. Can attention learn to focus on a marked token — and how does that compare to mean pooling and an RNN?

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. Scaled Dot-Product Attention

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Each query vector compares itself to all keys via dot product; softmax produces a
convex combination of value vectors. Scaling by $\sqrt{d_k}$ prevents dot products
from growing large (which would push softmax into saturation).

In [2]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


class ScaledDotProductAttention:
    def __init__(self):
        self.cache = {}

    def forward(self, Q, K, V, mask=None):
        d_k = Q.shape[-1]
        scores = Q @ np.swapaxes(K, -2, -1) / np.sqrt(d_k)
        if mask is not None:
            scores = np.where(mask, scores, -1e9)
        weights = softmax(scores, axis=-1)
        out = weights @ V
        self.cache = dict(Q=Q, K=K, V=V, weights=weights, mask=mask, d_k=d_k)
        return out, weights

    def backward(self, d_out):
        Q, K, V, weights, mask, d_k = self.cache.values()
        scale = 1.0 / np.sqrt(d_k)
        dW = d_out @ np.swapaxes(V, -2, -1)
        dV = np.swapaxes(weights, -2, -1) @ d_out
        dscores = weights * (dW - (weights * dW).sum(axis=-1, keepdims=True))
        if mask is not None:
            dscores = np.where(mask, dscores, 0.0)
        dQ = dscores @ K * scale
        dK = np.swapaxes(dscores, -2, -1) @ Q * scale
        return dQ, dK, dV


class SelfAttention:
    def __init__(self, d_model, seed=0):
        rng = np.random.RandomState(seed)
        s = np.sqrt(2.0 / d_model)
        self.Wq = rng.randn(d_model, d_model) * s
        self.Wk = rng.randn(d_model, d_model) * s
        self.Wv = rng.randn(d_model, d_model) * s
        self.Wo = rng.randn(d_model, d_model) * s
        self.attn = ScaledDotProductAttention()
        self.cache = {}

    def forward(self, X, mask=None):
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv
        ctx, weights = self.attn.forward(Q, K, V, mask=mask)
        out = ctx @ self.Wo
        self.cache = dict(X=X, Q=Q, K=K, V=V, ctx=ctx)
        return out, weights

    def backward(self, d_out):
        X, Q, K, V, ctx = self.cache.values()
        n, T, d = X.shape
        dctx = d_out @ self.Wo.T
        dWo = ctx.reshape(-1, d).T @ d_out.reshape(-1, d) / n
        dQ, dK, dV = self.attn.backward(dctx)
        flat_X = X.reshape(-1, d)
        dWq = flat_X.T @ dQ.reshape(-1, d) / (n * T)
        dWk = flat_X.T @ dK.reshape(-1, d) / (n * T)
        dWv = flat_X.T @ dV.reshape(-1, d) / (n * T)
        dX = dQ @ self.Wq.T + dK @ self.Wk.T + dV @ self.Wv.T
        return dict(Wq=dWq, Wk=dWk, Wv=dWv, Wo=dWo, dX=dX)


print('ScaledDotProductAttention and SelfAttention defined.')

ScaledDotProductAttention and SelfAttention defined.


## 2. PyTorch Validation

We compare forward pass and gradients against `torch.nn.functional.scaled_dot_product_attention`.

In [3]:
N, T, D = 2, 5, 8
Q = np.random.RandomState(0).randn(N, T, D)
K = np.random.RandomState(1).randn(N, T, D)
V = np.random.RandomState(2).randn(N, T, D)
d_out = np.random.RandomState(3).randn(N, T, D)

attn = ScaledDotProductAttention()
out, w = attn.forward(Q, K, V)
dQ, dK, dV = attn.backward(d_out)

Qt = torch.from_numpy(Q).requires_grad_(True)
Kt = torch.from_numpy(K).requires_grad_(True)
Vt = torch.from_numpy(V).requires_grad_(True)
out_t = F.scaled_dot_product_attention(Qt, Kt, Vt, is_causal=False)
(out_t * torch.from_numpy(d_out)).sum().backward()

print('Scaled dot-product attention validation:')
print(f'  forward max diff: {np.max(np.abs(out - out_t.detach().numpy())):.2e}')
print(f'  dQ max diff:      {np.max(np.abs(dQ - Qt.grad.numpy())):.2e}')
print(f'  dK max diff:      {np.max(np.abs(dK - Kt.grad.numpy())):.2e}')
print(f'  dV max diff:      {np.max(np.abs(dV - Vt.grad.numpy())):.2e}')

Scaled dot-product attention validation:
  forward max diff: 2.22e-16
  dQ max diff:      3.33e-16
  dK max diff:      4.44e-16
  dV max diff:      2.22e-16


## 3. Causal Masking

Autoregressive models (GPT-style) apply a **causal mask** so position $i$ cannot attend
to positions $j > i$. Without the mask, position 0 can leak information from the future.

In [4]:
attn = ScaledDotProductAttention()
X = np.random.RandomState(0).randn(1, 6, 8)
Q = K = V = X
_, w_full = attn.forward(Q, K, V)
causal = np.tril(np.ones((6, 6), dtype=bool))
_, w_causal = attn.forward(Q, K, V, mask=causal)

print(f'Position-0 attends to future (no mask):  {w_full[0, 0, 1:].sum():.3f}')
print(f'Position-0 attends to future (causal):   {w_causal[0, 0, 1:].sum():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(w_full[0], cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Full attention')
axes[1].imshow(w_causal[0], cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Causal (lower-triangular) mask')
for ax in axes:
    ax.set_xlabel('key position')
    ax.set_ylabel('query position')
plt.tight_layout()
plt.savefig('causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()

Position-0 attends to future (no mask):  0.066
Position-0 attends to future (causal):   0.000


## 4. Attention-Based Sequence Classifier

**Task:** each sequence has length $T$. All positions hold a padding token (9); one
random position holds the class digit (0–8). The model must classify the marked digit.

A **learnable query** attends over key/value projections of embedded tokens. We compare:
- **Attention pool** — softmax weights pick the marked position
- **Mean pool** — averages all positions (works here because padding is constant, but
  provides no interpretability)
- **Vanilla RNN** — must carry the marked digit through many noisy steps (Topic 10/11 failure mode)

In [5]:
PAD = 9


class Adam:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None:
            self.m = {k: np.zeros_like(p) for k, p in params.items()}
            self.v = {k: np.zeros_like(p) for k, p in params.items()}
        self.t += 1
        for k in params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k] ** 2
            m_hat = self.m[k] / (1 - self.beta1 ** self.t)
            v_hat = self.v[k] / (1 - self.beta2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class AttentionClassifier:
    def __init__(self, vocab=10, d=32, seed=0):
        rng = np.random.RandomState(seed)
        self.d = d
        self.embed = rng.randn(vocab, d) * 0.1
        self.Wk = rng.randn(d, d) * np.sqrt(2.0 / d)
        self.Wv = rng.randn(d, d) * np.sqrt(2.0 / d)
        self.q0 = rng.randn(d) * 0.1
        self.Wout = rng.randn(d, 9) * np.sqrt(2.0 / d)
        self.bout = np.zeros(9)
        self.attn = ScaledDotProductAttention()
        self.cache = {}

    def forward(self, x):
        H = self.embed[x]
        Q = np.tile(self.q0, (len(x), 1, 1))
        K = H @ self.Wk
        V = H @ self.Wv
        ctx, weights = self.attn.forward(Q, K, V)
        h = ctx[:, 0]
        logits = h @ self.Wout + self.bout
        self.cache = dict(x=x, H=H, K=K, V=V, ctx=ctx, h=h)
        return logits, weights[:, 0]

    def backward(self, d_logits):
        x, H, K, V, ctx, h = self.cache.values()
        n = len(x)
        dWout = h.T @ d_logits / n
        dbout = d_logits.mean(0)
        dh = d_logits @ self.Wout.T
        dctx = np.zeros_like(ctx)
        dctx[:, 0] = dh
        Q = np.tile(self.q0, (n, 1, 1))
        dQ, dK, dV = self.attn.backward(dctx)
        dq0 = dQ.sum(axis=0).reshape(-1)
        dWk = H.reshape(-1, self.d).T @ dK.reshape(-1, self.d) / (n * H.shape[1])
        dWv = H.reshape(-1, self.d).T @ dV.reshape(-1, self.d) / (n * H.shape[1])
        dH = dK @ self.Wk.T + dV @ self.Wv.T
        dembed = np.zeros_like(self.embed)
        for i in range(n):
            for t in range(x.shape[1]):
                dembed[x[i, t]] += dH[i, t]
        dembed /= n
        return dict(Wout=dWout, bout=dbout, q0=dq0 / n, Wk=dWk, Wv=dWv, embed=dembed)

    def param_dict(self):
        return dict(embed=self.embed, Wk=self.Wk, Wv=self.Wv, q0=self.q0,
                    Wout=self.Wout, bout=self.bout)

    def step(self, x, y, opt):
        logits, w = self.forward(x)
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        opt.step(self.param_dict(), self.backward(dlogits))
        return loss, w

    def accuracy(self, x, y):
        return float((self.forward(x)[0].argmax(1) == y).mean())


class MeanPoolClassifier:
    def __init__(self, vocab=10, d=32, seed=0):
        rng = np.random.RandomState(seed)
        self.embed = rng.randn(vocab, d) * 0.1
        self.Wout = rng.randn(d, 9) * np.sqrt(2.0 / d)
        self.bout = np.zeros(9)

    def forward(self, x):
        return self.embed[x].mean(axis=1) @ self.Wout + self.bout

    def step(self, x, y, opt):
        H = self.embed[x]
        h = H.mean(axis=1)
        logits = h @ self.Wout + self.bout
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        dWout = h.T @ dlogits / n
        dbout = dlogits.mean(0)
        dh = dlogits @ self.Wout.T
        dH = np.broadcast_to(dh[:, None, :], H.shape) / H.shape[1]
        dembed = np.zeros_like(self.embed)
        for i in range(n):
            for t in range(x.shape[1]):
                dembed[x[i, t]] += dH[i, t]
        dembed /= n
        opt.step(dict(embed=self.embed, Wout=self.Wout, bout=self.bout),
                 dict(embed=dembed, Wout=dWout, bout=dbout))
        return loss

    def accuracy(self, x, y):
        return float((self.forward(x).argmax(1) == y).mean())


class VanillaRNNClassifier:
    def __init__(self, vocab=10, hidden=32, seed=0):
        rng = np.random.RandomState(seed)
        self.hidden = hidden
        self.embed = rng.randn(vocab, hidden) * 0.1
        self.Whh = rng.randn(hidden, hidden) * 0.9
        self.Wy = rng.randn(hidden, 9) * np.sqrt(2.0 / hidden)
        self.by = np.zeros(9)

    def forward(self, x):
        n, T = x.shape
        h = np.zeros((n, self.hidden))
        for t in range(T):
            h = np.tanh(self.embed[x[:, t]] + h @ self.Whh)
        return h @ self.Wy + self.by

    def step(self, x, y, opt):
        n, T = x.shape
        hs = []
        h = np.zeros((n, self.hidden))
        for t in range(T):
            hs.append(h.copy())
            h = np.tanh(self.embed[x[:, t]] + h @ self.Whh)
        logits = h @ self.Wy + self.by
        probs = softmax(logits)
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dWy = h.T @ dlogits
        dby = dlogits.sum(0)
        dh = dlogits @ self.Wy.T
        dWhh = np.zeros_like(self.Whh)
        dembed = np.zeros_like(self.embed)
        for t in reversed(range(T)):
            dh_raw = dh * (1 - h ** 2)
            dWhh += hs[t].T @ dh_raw / n
            for i in range(n):
                dembed[x[i, t]] += dh_raw[i] / n
            dh = dh_raw @ self.Whh.T
        opt.step(dict(embed=self.embed, Whh=self.Whh, Wy=self.Wy, by=self.by),
                 dict(embed=dembed, Whh=dWhh, Wy=dWy, by=dby))
        return loss

    def accuracy(self, x, y):
        return float((self.forward(x).argmax(1) == y).mean())


def make_marker_data(n, T, seed=0):
    rng = np.random.RandomState(seed)
    X = np.full((n, T), PAD, dtype=int)
    y = rng.randint(0, 9, size=n)
    pos = rng.randint(0, T, size=n)
    for i in range(n):
        X[i, pos[i]] = y[i]
    return X, y, pos


def train_model(model, X, y, epochs=120, seed=0):
    opt = Adam(lr=0.01)
    rng = np.random.RandomState(seed)
    for _ in range(epochs):
        idx = rng.permutation(len(X))
        for i in range(0, len(X), 64):
            sl = idx[i:i + 64]
            if hasattr(model, 'attn'):
                model.step(X[sl], y[sl], opt)
            else:
                model.step(X[sl], y[sl], opt)


print('Classifiers and data helpers defined.')

Classifiers and data helpers defined.


## 5. Marker Task: Attention vs Mean Pool vs RNN

We train all three models at sequence lengths $T \in \{10, 20, 40\}$ and measure
test accuracy plus **average attention weight at the true marker position**.

In [6]:
SEEDS = [0, 1, 2]
rows = []
for T in [10, 20, 40]:
    acc_a, acc_m, acc_r, hits = [], [], [], []
    for s in SEEDS:
        X, y, pos = make_marker_data(3000, T, seed=s)
        Xtr, Xte, ytr, yte, _, pte = train_test_split(
            X, y, pos, test_size=0.2, random_state=0)
        att = AttentionClassifier(seed=s)
        mean = MeanPoolClassifier(seed=s)
        rnn = VanillaRNNClassifier(seed=s)
        train_model(att, Xtr, ytr, epochs=120, seed=s)
        train_model(mean, Xtr, ytr, epochs=120, seed=s)
        train_model(rnn, Xtr, ytr, epochs=120, seed=s)
        _, w = att.forward(Xte)
        acc_a.append(att.accuracy(Xte, yte))
        acc_m.append(mean.accuracy(Xte, yte))
        acc_r.append(rnn.accuracy(Xte, yte))
        hits.append(float(np.mean([w[i, pte[i]] for i in range(len(Xte))])))
    rows.append((T, np.mean(acc_a), np.mean(acc_m), np.mean(acc_r), np.mean(hits)))

print(f"{'T':>4} {'attention':>10} {'mean pool':>10} {'RNN':>8} {'attn@mark':>10}")
for T, aa, am, ar, hit in rows:
    print(f"{T:4d} {aa:10.3f} {am:10.3f} {ar:8.3f} {hit:10.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
Ts = [r[0] for r in rows]
ax.plot(Ts, [r[1] for r in rows], '-o', label='attention')
ax.plot(Ts, [r[2] for r in rows], '-s', label='mean pool')
ax.plot(Ts, [r[3] for r in rows], '-^', label='vanilla RNN')
ax.set_xlabel('sequence length T')
ax.set_ylabel('test accuracy')
ax.set_title('Marker-digit classification (3 seeds mean)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('marker_task_accuracy.png', dpi=100, bbox_inches='tight')
plt.show()

   T  attention  mean pool      RNN  attn@mark
  10      1.000      1.000    0.189      0.999
  20      1.000      1.000    0.186      1.000
  40      1.000      1.000    0.124      1.000


## 6. Visualizing Learned Attention Weights

For a trained attention classifier, weights should peak at the marked position
(near 1.0) instead of spreading uniformly ($1/T$).

In [7]:
T = 20
X, y, pos = make_marker_data(3000, T, seed=0)
Xtr, Xte, ytr, yte, _, pte = train_test_split(X, y, pos, test_size=0.2, random_state=0)
att = AttentionClassifier(seed=0)
train_model(att, Xtr, ytr, epochs=120, seed=0)
_, weights = att.forward(Xte[:6])

fig, axes = plt.subplots(2, 3, figsize=(9, 4))
for j, ax in enumerate(axes.flat):
    ax.bar(range(T), weights[j], color='steelblue')
    ax.axvline(pte[j], color='crimson', ls='--', lw=1.5, label='marker')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'sample {j}, y={yte[j]}, marker@{pte[j]}')
    ax.set_xlabel('position')
    if j == 0:
        ax.set_ylabel('attention weight')
plt.suptitle(f'Learned attention weights (uniform baseline = {1/T:.3f})')
plt.tight_layout()
plt.savefig('attention_weights_bars.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mean weight at marker: {np.mean([weights[j, pte[j]] for j in range(6)]):.3f}')
print(f'Uniform baseline:      {1/T:.3f}')

Mean weight at marker: 1.000
Uniform baseline:      0.050
